# Weight Initialization in Deep Neural Networks
## A beginner-friendly, visual lab with ReLU, Leaky ReLU, tanh, sigmoid, and softmax

**What this notebook will teach you**

1. Why the first weights in a neural network matter.
2. What `fan_in` and `fan_out` mean.
3. Why zero and constant-one initialization cause problems.
4. The difference between random normal, random uniform, Xavier/Glorot, and He/Kaiming initialization.
5. How initialization interacts with ReLU, Leaky ReLU, tanh, and sigmoid.
6. How softmax turns output logits into class probabilities.
7. How to compare training curves, accuracy, weight changes, and gradient sizes.

We will train small neural networks on the **two-moons classification dataset**. It is small, visual, and useful for seeing when a network learns a curved decision boundary.

> **Source note:** This notebook is a detailed, original-language study guide based on the concepts in the GeeksforGeeks article linked below. It does not reproduce the entire copyrighted article verbatim. It summarizes the article's sections, gives working examples, and adds explanations and experiments. The source article remains the place to read its original wording.

## 1. Source article and what it covers

**Reference article:** [Weight Initialization Techniques for Deep Neural Networks — GeeksforGeeks](https://www.geeksforgeeks.org/machine-learning/weight-initialization-techniques-for-deep-neural-networks/)

The article's relevant sections cover:

- **Terminology:** `fan_in` counts connections entering a layer; `fan_out` counts connections leaving it.
- **Zero initialization:** every weight begins at zero. Neurons in the same layer start identically and receive identical updates, so they fail to learn different features.
- **Random initialization:** weights are sampled rather than assigned a common constant. Random values still need an appropriate scale.
- **Random normal:** sample from a normal distribution, usually described by its mean and standard deviation.
- **Random uniform:** sample values between a minimum and maximum.
- **Xavier/Glorot uniform and normal:** scale initial weights using both `fan_in` and `fan_out`; commonly suitable for tanh and often sigmoid.
- **He/Kaiming uniform and normal:** scale initial weights using `fan_in`; commonly suitable for ReLU-family activations.
- **Practical implementation:** the article demonstrates initializer objects in TensorFlow/Keras.
- **Conclusion:** initialization should be selected with the activation function and network structure in mind.

The experiments below extend these concepts by comparing initializers on exactly the same dataset, architecture, optimizer, and number of epochs.

### A note about formula rendering

The article's displayed/extracted formulas have a few inconsistencies (for example, the random-uniform formula appears as a normal distribution in one place, and the Glorot-normal / He-uniform expressions are easy to misread). To avoid teaching an incorrect formula, this notebook uses the official Keras initializer definitions:

- [Keras initializer documentation](https://keras.io/api/layers/initializers/)
- [PyTorch initialization reference](https://docs.pytorch.org/docs/stable/nn.init/)

The rest of this notebook explains the standard formulas explicitly.

## 2. What is weight initialization?

A dense layer computes

\[
z = Wa + b,\qquad a_{\mathrm{next}} = g(z)
\]

where:

- \(a\) is the vector of activations from the previous layer,
- \(W\) is the weight matrix,
- \(b\) is the bias vector,
- \(g\) is the activation function.

Training does not usually start with useful weights. At initialization, we choose the starting values of \(W\) (and usually start biases at zero).

A bad starting scale can make values or gradients shrink towards zero (**vanishing**) or grow excessively (**exploding**). Constant initialization also creates a separate problem: identical neurons can stay identical.

### `fan_in` and `fan_out`

For a Dense layer:

- `fan_in` = number of incoming features.
- `fan_out` = number of neurons produced by that layer.

For a layer with 2 inputs and 16 output neurons, `fan_in = 2`, `fan_out = 16`. For the next 16-to-16 layer, both are 16.

## 3. Initialization formulas

Let \(n_{\mathrm{in}}\) be `fan_in` and \(n_{\mathrm{out}}\) be `fan_out`.

### A. Zero and constant initialization

\[
W_{ij}=0 \qquad \text{(zero initialization)}
\]

\[
W_{ij}=1 \qquad \text{(constant-one initialization)}
\]

These are useful as **negative demonstrations**, not as normal defaults for all trainable weights. Because neurons in a layer start with the same incoming weights, they compute the same output and often get the same gradient.

### B. Random normal and random uniform

\[
W_{ij}\sim\mathcal{N}(\mu,\sigma^2)
\]

Here \(\mu\) is the mean and \(\sigma\) is the standard deviation.

\[
W_{ij}\sim U(a,b)
\]

Here \(a\) and \(b\) are the lower and upper bounds. Randomness alone does not guarantee good initialization: values sampled from an unnecessarily large or non-zero-centred range can still cause saturation or unstable activations.

### C. Xavier / Glorot initialization

**Xavier uniform:**

\[
W_{ij}\sim U\left(-\sqrt{\frac{6}{n_{\mathrm{in}}+n_{\mathrm{out}}}},
+\sqrt{\frac{6}{n_{\mathrm{in}}+n_{\mathrm{out}}}}\right)
\]

**Xavier normal:**

\[
W_{ij}\sim\mathcal{N}\left(0,\frac{2}{n_{\mathrm{in}}+n_{\mathrm{out}}}\right)
\]

The second argument in the normal formula above is the **variance**. Its standard deviation is \(\sqrt{2/(n_{\mathrm{in}}+n_{\mathrm{out}})}\).

Xavier scales by both the number of incoming and outgoing connections. It is a common starting choice for tanh and sigmoid networks.

### D. He / Kaiming initialization

**He normal:**

\[
W_{ij}\sim\mathcal{N}\left(0,\frac{2}{n_{\mathrm{in}}}\right)
\]

Its standard deviation is \(\sqrt{2/n_{\mathrm{in}}}\).

**He uniform:**

\[
W_{ij}\sim U\left(-\sqrt{\frac{6}{n_{\mathrm{in}}}},
+\sqrt{\frac{6}{n_{\mathrm{in}}}}\right)
\]

He initialization is designed to compensate for the reduction in signal caused by ReLU-like activations. For Leaky ReLU with negative slope \(\alpha\), a commonly used He-style variance is

\[
\operatorname{Var}(W)=\frac{2}{(1+\alpha^2)n_{\mathrm{in}}}.
\]

In this practical lab, we use Keras `HeNormal()` for both ReLU and Leaky ReLU; with a small slope such as 0.01, the standard He choice is a close practical approximation.

## 4. The activations we will compare

- **ReLU:** \(g(x)=\max(0,x)\). Negative inputs become zero.
- **Leaky ReLU:** \(g(x)=x\) when \(x>0\), and \(\alpha x\) otherwise. It keeps a small negative-side slope.
- **tanh:** outputs values between -1 and 1; its derivative is strongest near zero and small when saturated.
- **sigmoid:** outputs values between 0 and 1; its derivative is at most 0.25 and becomes very small when the input is far from zero.
- **softmax:** usually used in the final classification layer to turn a vector of logits into probabilities that sum to 1. It is an output transformation, not typically the hidden-layer activation whose initializer we choose.

The key practical pairing used in this notebook is:

| Layer activation | Recommended hidden-layer initializer |
|---|---|
| ReLU | He / Kaiming |
| Leaky ReLU | He-style / Kaiming |
| tanh | Xavier / Glorot |
| sigmoid | Xavier / Glorot |
| softmax output | Often Glorot for its Dense logits layer; the hidden-layer initializer still depends on the hidden activation |

These are sensible defaults, not universal laws. Normalization, residual connections, depth, optimizer, and data scale also affect training.

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available GPU devices:", tf.config.list_physical_devices("GPU"))

## 5. Make the dataset

We will use `make_moons`, a small two-class dataset with curved shapes. It is generated locally by scikit-learn, so the notebook does not need to download a dataset file.

We standardize the input features because initialization experiments are easier to interpret when the input scale is reasonable.

In [ ]:
X, y = make_moons(n_samples=1400, noise=0.22, random_state=SEED)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=SEED,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype("float32")
X_test = scaler.transform(X_test_raw).astype("float32")
y_train = y_train.astype("int32")
y_test = y_test.astype("int32")

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))
print("Input features:", X_train.shape[1])
print("Classes:", np.unique(y_train))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=18, alpha=0.75)
plt.title("Training data: two moons")
plt.xlabel("Standardized feature 1")
plt.ylabel("Standardized feature 2")
plt.grid(alpha=0.2)
plt.show()

## 6. Plot the activation functions

This helps explain why initialization differs. In particular, sigmoid and tanh can saturate for large positive or negative inputs; ReLU sets negative values to zero.

In [ ]:
x = np.linspace(-5, 5, 400)
relu = np.maximum(0, x)
alpha = 0.01
leaky_relu = np.where(x > 0, x, alpha * x)
tanh = np.tanh(x)
sigmoid = 1 / (1 + np.exp(-x))

plt.figure(figsize=(9, 5))
plt.plot(x, relu, label="ReLU")
plt.plot(x, leaky_relu, label="Leaky ReLU (alpha=0.01)")
plt.plot(x, tanh, label="tanh")
plt.plot(x, sigmoid, label="sigmoid")
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.ylim(-1.5, 5.2)
plt.title("Activation functions")
plt.xlabel("Input")
plt.ylabel("Activation output")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 7. Implement the initializers in Keras

We will compare five conditions:

1. **Zeros:** every Dense layer's kernel starts at 0.
2. **Ones:** every Dense layer's kernel starts at 1.
3. **Random normal, standard deviation 1:** a deliberately simple, potentially poorly scaled random baseline.
4. **Random uniform [0, 1]:** another basic random baseline. Since values are all non-negative and not centred on zero, this may be a poor choice for hidden weights.
5. **Recommended:** He normal for ReLU/Leaky ReLU hidden layers; Glorot normal for tanh/sigmoid hidden layers; Glorot uniform for the final softmax classifier.

For a fair comparison, **all biases are initialized to zero**. In the zero/one/random-baseline conditions, the chosen kernel initializer is used for the output layer too. The recommended condition uses Glorot uniform for the output Dense layer because softmax is the classifier head.

This makes the poor-initialization conditions intentionally easy to identify. We will also inspect gradients before training.

In [ ]:
ACTIVATIONS = ["relu", "leaky_relu", "tanh", "sigmoid"]
INITIALIZERS = [
    "Zeros",
    "Ones",
    "Random Normal (std=1)",
    "Random Uniform [0,1]",
    "Recommended"
]

def make_kernel_initializer(initializer_name, activation_name, seed):
    if initializer_name == "Zeros":
        return tf.keras.initializers.Zeros()
    if initializer_name == "Ones":
        return tf.keras.initializers.Ones()
    if initializer_name == "Random Normal (std=1)":
        return tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0, seed=seed)
    if initializer_name == "Random Uniform [0,1]":
        return tf.keras.initializers.RandomUniform(minval=0.0, maxval=1.0, seed=seed)
    if initializer_name == "Recommended":
        if activation_name in ("relu", "leaky_relu"):
            return tf.keras.initializers.HeNormal(seed=seed)
        return tf.keras.initializers.GlorotNormal(seed=seed)
    raise ValueError(f"Unknown initializer: {initializer_name}")

def build_model(activation_name, initializer_name, seed=SEED, units=24):
    tf.keras.utils.set_random_seed(seed)

    inputs = tf.keras.Input(shape=(2,), name="input_features")

    init_1 = make_kernel_initializer(initializer_name, activation_name, seed + 1)
    init_2 = make_kernel_initializer(initializer_name, activation_name, seed + 2)

    h = tf.keras.layers.Dense(
        units,
        kernel_initializer=init_1,
        bias_initializer="zeros",
        name="hidden_dense_1"
    )(inputs)

    if activation_name == "leaky_relu":
        h = tf.keras.layers.LeakyReLU(negative_slope=0.01, name="hidden_activation_1")(h)
    else:
        h = tf.keras.layers.Activation(activation_name, name="hidden_activation_1")(h)

    h = tf.keras.layers.Dense(
        units,
        kernel_initializer=init_2,
        bias_initializer="zeros",
        name="hidden_dense_2"
    )(h)

    if activation_name == "leaky_relu":
        h = tf.keras.layers.LeakyReLU(negative_slope=0.01, name="hidden_activation_2")(h)
    else:
        h = tf.keras.layers.Activation(activation_name, name="hidden_activation_2")(h)

    if initializer_name == "Recommended":
        output_initializer = tf.keras.initializers.GlorotUniform(seed=seed + 3)
    else:
        output_initializer = make_kernel_initializer(
            initializer_name, activation_name, seed + 3
        )

    outputs = tf.keras.layers.Dense(
        2,
        activation="softmax",
        kernel_initializer=output_initializer,
        bias_initializer="zeros",
        name="softmax_output"
    )(h)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001,
            clipnorm=1.0
        ),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )
    return model

print("Model builder ready.")

## 8. Look at the actual weights before training

The histograms let us see the initial distributions. A good initializer does not try to make every weight large; it chooses a scale based on the layer size and activation.

In [ ]:
plt.figure(figsize=(10, 5))

examples = [
    ("Zeros", "tanh"),
    ("Ones", "relu"),
    ("Random Normal (std=1)", "relu"),
    ("Recommended", "relu"),
]

for i, (init_name, act_name) in enumerate(examples, start=1):
    model = build_model(act_name, init_name, seed=SEED + i)
    weights = model.get_layer("hidden_dense_1").get_weights()[0].ravel()
    plt.subplot(2, 2, i)
    plt.hist(weights, bins=24)
    plt.title(f"{init_name}\\n{act_name}, first hidden layer")
    plt.xlabel("Weight value")
    plt.ylabel("Count")

plt.tight_layout()
plt.show()

## 9. Inspect activations before training

Healthy initialization aims to keep activations in useful ranges. This is not a rule that every activation must have mean 0 and standard deviation 1; it is a way to spot dead or saturated layers.

- With zero weights, ReLU and tanh hidden activations start at exactly zero.
- With zero weights and sigmoid, hidden activations start at 0.5, identical for all examples and neurons.
- With constant-one weights, neurons start identically within each layer. Large pre-activations can push sigmoid/tanh towards saturation or make ReLU activations large.

In [ ]:
def activation_statistics(activation_name, initializer_name):
    model = build_model(activation_name, initializer_name, seed=123)
    probe = tf.keras.Model(
        inputs=model.input,
        outputs=[
            model.get_layer("hidden_activation_1").output,
            model.get_layer("hidden_activation_2").output
        ]
    )
    a1, a2 = probe(X_train[:256], training=False)

    def stats(a):
        a = a.numpy()
        return {
            "mean": float(np.mean(a)),
            "std": float(np.std(a)),
            "fraction near zero": float(np.mean(np.abs(a) < 1e-6)),
            "fraction |a| > 0.95": float(np.mean(np.abs(a) > 0.95))
        }

    return stats(a1), stats(a2)

rows = []
for act in ACTIVATIONS:
    for init_name in ["Zeros", "Ones", "Recommended"]:
        s1, s2 = activation_statistics(act, init_name)
        rows.append({
            "Activation": act,
            "Initialization": init_name,
            "Layer 1 mean": s1["mean"],
            "Layer 1 std": s1["std"],
            "Layer 1 near-zero fraction": s1["fraction near zero"],
            "Layer 1 near-saturated fraction": s1["fraction |a| > 0.95"],
            "Layer 2 mean": s2["mean"],
            "Layer 2 std": s2["std"],
            "Layer 2 near-zero fraction": s2["fraction near zero"],
            "Layer 2 near-saturated fraction": s2["fraction |a| > 0.95"],
        })

activation_stats_df = pd.DataFrame(rows)
display(activation_stats_df.round(4))

## 10. Compare gradient sizes at the very beginning

Before any training, calculate gradients for one batch. The norm is a single number summarizing gradient magnitude for a layer's weights.

In the **all-zero ReLU/tanh network**, hidden activations are zero, so output-kernel gradients are zero too. Since downstream kernels are also zero, no useful gradient reaches the hidden kernels. The output bias can still change; this is not the same as the entire model receiving no gradient at all.

For sigmoid, zero weights produce 0.5 activations, so the output kernel can get a gradient immediately. But the hidden neurons begin identically, so they still have a symmetry problem.

This gradient inspection is one reason we should avoid the blanket statement that a zero-initialized model has absolutely no parameter updates: output biases can update, and behaviour depends on activation and architecture.

In [ ]:
def initial_gradient_report(activation_name, initializer_name, seed=77):
    model = build_model(activation_name, initializer_name, seed=seed)
    xb = tf.convert_to_tensor(X_train[:128])
    yb = tf.convert_to_tensor(y_train[:128])

    with tf.GradientTape() as tape:
        probs = model(xb, training=True)
        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(yb, probs)
        )

    grads = tape.gradient(loss, model.trainable_variables)
    # Map gradients by variable identity; avoid comparing tensor variables by value.
    gradient_by_id = {
        id(variable): grad
        for variable, grad in zip(model.trainable_variables, grads)
        if grad is not None
    }

    # Sum kernel-gradient norms by layer for easy reading.
    rows = []
    for layer_name in ["hidden_dense_1", "hidden_dense_2", "softmax_output"]:
        layer = model.get_layer(layer_name)
        kernel_grad = gradient_by_id.get(id(layer.kernel))
        bias_grad = gradient_by_id.get(id(layer.bias))
        rows.append({
            "Activation": activation_name,
            "Initialization": initializer_name,
            "Layer": layer_name,
            "Kernel gradient norm": float(tf.linalg.global_norm([kernel_grad]).numpy()) if kernel_grad is not None else 0.0,
            "Bias gradient norm": float(tf.linalg.global_norm([bias_grad]).numpy()) if bias_grad is not None else 0.0
        })
    return rows

grad_rows = []
for act in ["relu", "tanh", "sigmoid"]:
    for init_name in ["Zeros", "Recommended"]:
        grad_rows.extend(initial_gradient_report(act, init_name))

gradient_df = pd.DataFrame(grad_rows)
display(gradient_df.round(8))

## 11. Train every activation–initializer combination

This is the main experiment. To make it a fair educational comparison, every model sees:

- the same train/test split,
- the same two-hidden-layer architecture,
- the same batch size and epoch count,
- the same optimizer and learning rate,
- the same loss function.

Only the hidden activation and initialization strategy change. Randomness and small datasets can affect the exact results, so focus on the overall pattern rather than expecting fixed accuracy values.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64

histories = {}
trained_models = {}
summary_rows = []

start_all = time.time()

for act_index, act in enumerate(ACTIVATIONS):
    for init_index, init_name in enumerate(INITIALIZERS):
        run_seed = 1000 + act_index * 100 + init_index

        tf.keras.backend.clear_session()
        model = build_model(act, init_name, seed=run_seed)

        initial_kernels = {
            layer_name: model.get_layer(layer_name).get_weights()[0].copy()
            for layer_name in ["hidden_dense_1", "hidden_dense_2", "softmax_output"]
        }

        print(f"Training: activation={act:10s} | initializer={init_name}")

        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )

        test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

        delta_norms = {}
        for layer_name, initial_w in initial_kernels.items():
            final_w = model.get_layer(layer_name).get_weights()[0]
            delta_norms[layer_name] = float(np.linalg.norm(final_w - initial_w))

        key = (act, init_name)
        histories[key] = history.history
        trained_models[key] = model

        summary_rows.append({
            "Activation": act,
            "Initialization": init_name,
            "Train accuracy (%)": round(history.history["accuracy"][-1] * 100, 2),
            "Test accuracy (%)": round(test_accuracy * 100, 2),
            "Test loss": round(float(test_loss), 4),
            "Change in hidden layer 1 weights": round(delta_norms["hidden_dense_1"], 6),
            "Change in hidden layer 2 weights": round(delta_norms["hidden_dense_2"], 6),
            "Change in output weights": round(delta_norms["softmax_output"], 6)
        })

        # Free backend graph/session state while retaining Python history/model references.
        del model

print(f"\\nTotal experiment time: {(time.time() - start_all) / 60:.1f} minutes")
summary_df = pd.DataFrame(summary_rows)
display(summary_df.sort_values(["Activation", "Test accuracy (%)"], ascending=[True, False]).reset_index(drop=True))

## 12. Plot the validation-accuracy curves

Each panel holds the activation fixed and compares initialization methods. A flat curve means the model is not improving much on the held-out data. A rising curve means it is learning a useful decision boundary.

A few practical caveats:

- Accuracy may fluctuate because we use a small dataset and finite training time.
- Initialization is not the only cause of poor learning; learning rate, activation saturation, architecture, and data also matter.
- A model can improve its accuracy through output bias changes without learning useful hidden features. That is why the notebook also reports hidden-weight changes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
axes = axes.ravel()

for ax, act in zip(axes, ACTIVATIONS):
    for init_name in INITIALIZERS:
        hist = histories[(act, init_name)]
        epochs = np.arange(1, len(hist["val_accuracy"]) + 1)
        ax.plot(epochs, np.array(hist["val_accuracy"]) * 100, label=init_name)

    ax.set_title(f"Hidden activation: {act}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation accuracy (%)")
    ax.grid(alpha=0.25)

axes[-1].legend(loc="lower right", fontsize=8)
plt.suptitle("Validation accuracy by initializer and activation", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

## 13. Plot the validation-loss curves

Loss gives more detail than accuracy: it measures how well the predicted probabilities match the correct labels. A model can have the same accuracy as another model but a worse loss if it is overly confident in incorrect predictions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.ravel()

for ax, act in zip(axes, ACTIVATIONS):
    for init_name in INITIALIZERS:
        hist = histories[(act, init_name)]
        epochs = np.arange(1, len(hist["val_loss"]) + 1)
        ax.plot(epochs, hist["val_loss"], label=init_name)

    ax.set_title(f"Hidden activation: {act}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation cross-entropy loss")
    ax.grid(alpha=0.25)

axes[-1].legend(loc="upper right", fontsize=8)
plt.suptitle("Validation loss by initializer and activation", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

## 14. Compare weight updates directly

An initialization can leave weights unchanged even while some biases update. The table below makes that visible.

- If a kernel's change is nearly zero, that layer's **weights** did not meaningfully update.
- Under all-zero ReLU/tanh initialization in this network, expect hidden kernels to remain unchanged because zero activations and zero downstream kernels block the learning signal.
- Sigmoid starts at 0.5 rather than 0, allowing the output kernel to receive gradients. However, zero initialization still makes neurons symmetric, so the network cannot develop a different feature in every neuron.
- Constant-one initialization can allow some updates, but identical neurons and poor activation scale can still limit learning.

In [ ]:
display(
    summary_df[
        [
            "Activation", "Initialization", "Test accuracy (%)",
            "Change in hidden layer 1 weights",
            "Change in hidden layer 2 weights",
            "Change in output weights"
        ]
    ].sort_values(["Activation", "Initialization"]).reset_index(drop=True)
)

## 15. Inspect the final softmax probabilities

Softmax converts logits \(z_i\) into probabilities:

\[
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}
\]

For a two-class model, the probabilities add to 1. If both logits are equal, softmax returns 0.5 for each class. A Dense layer's **weights still need an initializer**; softmax itself is the mathematical function applied to the logits.

We train with sparse categorical cross-entropy. In Keras, using `activation="softmax"` with `SparseCategoricalCrossentropy()` is a valid beginner-friendly pairing.

In [ ]:
example_logits = np.array([2.0, 1.0, 0.1])
example_exp = np.exp(example_logits - np.max(example_logits))  # stable softmax
example_probs = example_exp / example_exp.sum()

print("Example logits:", example_logits)
print("Softmax probabilities:", np.round(example_probs, 4))
print("Probability sum:", example_probs.sum())

plt.figure(figsize=(6, 4))
plt.bar(["Class A", "Class B", "Class C"], example_probs)
plt.ylim(0, 1)
plt.ylabel("Probability")
plt.title("Softmax turns logits into probabilities")
plt.grid(axis="y", alpha=0.2)
plt.show()

## 16. Visualize a learned decision boundary

A useful model should learn the curved pattern of the moons. Compare the recommended setup against zero initialization and constant-one initialization for the same activation.

If the output looks different on your run, inspect the corresponding test accuracy and curves. The result is measured by the model, not assumed in advance.

In [ ]:
def plot_decision_boundary(model, title, ax):
    x_min, x_max = X_train[:, 0].min() - 0.7, X_train[:, 0].max() + 0.7
    y_min, y_max = X_train[:, 1].min() - 0.7, X_train[:, 1].max() + 0.7
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 220),
        np.linspace(y_min, y_max, 220)
    )
    grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
    probs = model.predict(grid, verbose=0)[:, 1].reshape(xx.shape)

    ax.contourf(xx, yy, probs, levels=np.linspace(0, 1, 15), alpha=0.55)
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, s=12, edgecolors="k", linewidths=0.15)
    ax.set_title(title)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
act_to_show = "relu"

for ax, init_name in zip(axes, ["Zeros", "Ones", "Recommended"]):
    model = trained_models[(act_to_show, init_name)]
    row = summary_df[
        (summary_df["Activation"] == act_to_show) &
        (summary_df["Initialization"] == init_name)
    ].iloc[0]
    plot_decision_boundary(
        model,
        f"{act_to_show} + {init_name}\\nTest accuracy: {row['Test accuracy (%)']}%",
        ax
    )

plt.tight_layout()
plt.show()

## 17. Read the results correctly

### What you should generally observe

**Zero initialization**
- All hidden neurons start with the same weights and same activations.
- With ReLU and tanh in this fully zero-initialized setup, hidden activations are zero. The output may adjust its bias, but the hidden kernels cannot start learning useful features.
- With sigmoid, zero pre-activations produce 0.5, so the output layer can receive a gradient. However, symmetry remains: hidden units start identical and receive identical updates.

**Constant-one initialization**
- Neurons in the same layer again start identically.
- ReLU can create very large activations through repeated sums.
- Tanh and sigmoid can saturate, where derivatives are small.
- Some parameters may still change, but the network starts with poor feature diversity.

**Random normal / random uniform baselines**
- Random initialization can break symmetry, but arbitrary scale is not the same as a well-chosen scale.
- Uniform values in [0, 1] are all positive and can shift pre-activations away from zero.
- A normal distribution with standard deviation 1 may be too large for some layers and activation functions.

**Recommended initialization**
- He normal is a good standard for ReLU and Leaky ReLU hidden layers.
- Glorot/Xavier normal is a good standard for tanh and often sigmoid hidden layers.
- These techniques aim to keep signal and gradient scales useful; they do not guarantee perfect learning or the highest possible accuracy on every dataset.

### A correction to a common oversimplification

It is not mathematically accurate to say **every** zero-initialized network will have literally zero parameter updates. Biases can change, and sigmoid can pass gradients differently from ReLU. The key principle is that identical initial weights preserve symmetry, so neurons fail to learn distinct features. This notebook shows the behaviour for a specific controlled architecture.

## 18. A compact reference table

| Technique | Formula / distribution | Common use |
|---|---|---|
| Zeros | \(W=0\) | Demonstration only for trainable hidden layers; causes symmetry |
| Ones / constant | \(W=c\) | Usually avoid for trainable hidden layers due symmetry |
| Random normal | \(\mathcal{N}(\mu,\sigma^2)\) | Only sensible if scale is chosen appropriately |
| Random uniform | \(U(a,b)\) | Only sensible if bounds are chosen appropriately |
| Xavier / Glorot uniform | \(U(-\sqrt{6/(fan_{in}+fan_{out})},+\sqrt{6/(fan_{in}+fan_{out})})\) | tanh/sigmoid as a common default |
| Xavier / Glorot normal | std \(\sqrt{2/(fan_{in}+fan_{out})}\) | tanh/sigmoid as a common default |
| He / Kaiming uniform | \(U(-\sqrt{6/fan_{in}},+\sqrt{6/fan_{in}})\) | ReLU-family hidden layers |
| He / Kaiming normal | std \(\sqrt{2/fan_{in}}\) | ReLU-family hidden layers |

### Practical mnemonic

- **ReLU → He**
- **Leaky ReLU → He-style / Kaiming**
- **tanh → Xavier / Glorot**
- **sigmoid → Xavier / Glorot**
- **softmax → initialize the logits-producing Dense layer sensibly; Glorot is a common default**

## 19. Further experiments

Change one thing at a time and rerun the notebook:

1. Change `EPOCHS` from 20 to 40.
2. Change `units=24` to 8, 32, or 64 in `build_model`.
3. Try `learning_rate=0.0005` or `0.003`.
4. Try `make_moons(noise=0.10)` and then `noise=0.35`.
5. Repeat experiments with 3–5 different random seeds and compare average accuracy; one run can be noisy.
6. Try a deeper network. Poor initialization often becomes more obvious as depth increases.
7. Try He uniform instead of He normal, or Glorot uniform instead of Glorot normal.

Do not judge an initializer only by the final accuracy of a single run. Also inspect the learning curves, initial activation statistics, gradient norms, and whether different neurons can learn different features.

## References

1. GeeksforGeeks, [Weight Initialization Techniques for Deep Neural Networks](https://www.geeksforgeeks.org/machine-learning/weight-initialization-techniques-for-deep-neural-networks/) — original concept outline and TensorFlow/Keras examples.
2. Keras, [Layer weight initializers](https://keras.io/api/layers/initializers/) — official formulas and initializer behaviour.
3. PyTorch, [Parameter initialization](https://docs.pytorch.org/docs/stable/nn.init.html) — additional definitions and formulas.

**End of lab.** You now have a runnable experiment as well as the theory behind the results.